## PMN Distance to Immune cells

#### Load Protein Data

In [2]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

# Load files on Evan's Laptop
# expr_orig = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\expression.csv", index_col=0)
# expr=expr_orig.transpose()
# metadata = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\metadata.csv", index_col=0)
# umap = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\umap.csv", index_col=0)

#Load files on Lab computer
expr_orig = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\expression.csv", index_col=0)
expr=expr_orig.transpose()
metadata = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\metadata.csv", index_col=0)
umap = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\umap.csv", index_col=0)

# Create AnnData object
adata = sc.AnnData(X=expr.values)

# Assign metadata
adata.obs = metadata
adata.var_names = expr.columns
adata.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
# adata.obsm["spatial"] = metadata[['x_FOV_px', 'y_FOV_px']].values  # adjust if needed
adata.obsm["spatial"] = metadata[['x_FOV_px']].assign(y_FOV_px = -metadata['y_FOV_px']).values
adata.obsm["X_umap"] = umap.values

C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import annd

### Exact Distances of PMN cells to other cell types 

This is probably not very helpful and not particullary useful in and of itself. However, this was requested by Huy so I will go along with it. What this does is measure the distance between PMN and the targeted cell types (more can be added easily) and this will return information like the average distance to the closest cell and the average distance to all cells. 

In [13]:
# Output. Each sample in order with the following columns Sample ID, PMN Count, Resonder Status, Average Min Distance CD4, Average Min Distance  CD8,  Average Min Distance  tumor cells,  Average Min Distance Treg (to start
import pandas as pd
import importlib
import my_functions
importlib.reload(my_functions)

# Define column names
columns = ['Sample ID', 'PMN Count', 'Responder Status', 'Avg Min CD4 Distance', 'Avg Min CD8 Distance', 'Avg Min Treg Distance', 'Average Min Tumor Distance', 'Avg CD4 Distance', 'Avg CD8 Distance', 'Avg Treg Distance', 'Average Tumor Distance']

# Create empty DataFrame with those columns
summary_matrix = pd.DataFrame(columns=columns)

for i in range(1,4):
    for j in range(1,26):
        
        # Data to add to matrix
        sample_id=f"c_{i}_{j}_"
        pmn_count=my_functions.pmn_counter(adata,sample_id)["PMN Count"]

        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue
        responder_status=my_functions.get_sample_info(adata,sample_id)["Response"]

        # CD4 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD4+T_cells")["Cell Count"] > 0:
            results_CD4=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD4+T_cells")
            avg_min_CD4=results_CD4["Average Mininum Distance"]
            avg_CD4=results_CD4["Average Distance"]
        else:
            avg_min_CD4=0
            avg_CD4=0

        # CD8 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD8+T_cells")["Cell Count"] > 0:
            results_CD8=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD8+T_cells")
            avg_min_CD8=results_CD8["Average Mininum Distance"]
            avg_CD8=results_CD8["Average Distance"]
        else:
            avg_min_CD8=0
            avg_CD8=0

        # Treg Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Treg")["Cell Count"] > 0:
            results_Treg=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Treg")
            avg_min_Treg=results_Treg["Average Mininum Distance"]
            avg_Treg=results_Treg["Average Distance"]
        else:
            avg_min_Treg=0
            avg_Treg=0

        # Tumor Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Tumor_cells")["Cell Count"] > 0:
            results_Tumor=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Tumor_cells")
            avg_min_tumor=results_Tumor["Average Mininum Distance"]
            avg_tumor=results_Tumor["Average Distance"]
        else:
            avg_min_tumor=0
            avg_tumor=0
            
        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue

        # Adding data to matrix
        summary_matrix.loc[len(summary_matrix)] = [sample_id, pmn_count, responder_status, avg_min_CD4, avg_min_CD8, avg_min_Treg, avg_min_tumor, avg_CD4, avg_CD8, avg_Treg, avg_tumor]
        
display(summary_matrix)
summary_matrix.to_excel("summary_matrix_2.xlsx", index=False)

[NbConvertApp] Converting notebook my_functions.ipynb to python
[NbConvertApp] Writing 24140 bytes to my_functions.py


,Sample ID,PMN Count,Responder Status,Avg Min CD4 Distance,Avg Min CD8 Distance,Avg Min Treg Distance,Average Min Tumor Distance,Avg CD4 Distance,Avg CD8 Distance,Avg Treg Distance,Average Tumor Distance
0,c_1_1_,55,R,273.972614,592.759764,631.634319,11.090332,525.352298,735.917035,651.116774,446.532664
1,c_1_3_,6,R,28.753049,170.031600,106.209140,11.824233,246.332199,351.629350,224.877578,330.041189
2,c_1_4_,12,R,52.528240,99.802765,73.496580,23.712366,234.948735,287.666555,248.699919,369.395953
3,c_1_5_,15,R,86.062535,150.281274,64.924672,15.375530,347.959190,392.597713,366.510778,400.521354
4,c_1_6_,23,R,19.640478,375.802924,74.626134,33.386715,99.011878,573.904537,74.626134,496.276498
...,...,...,...,...,...,...,...,...,...,...,...
61,c_3_16_,18,NR,75.778988,183.079156,64.614820,29.833396,302.747329,264.613239,294.534193,463.074163
62,c_3_17_,22,NR,63.788160,151.555074,100.891480,30.778431,336.877326,260.473369,317.286085,496.839089
63,c_3_18_,36,NR,55.022038,144.141265,64.205251,28.126310,419.253659,459.742630,409.799740,474.083958
64,c_3_19_,36,NR,48.176947,348.449298,62.653424,22.721432,370.779279,513.769397,328.502648,404.308327


In [31]:
summary_matrix.to_excel("summary_matrix_2.xlsx", index=False)

In [7]:
import my_functions
import importlib
importlib.reload(my_functions)


[NbConvertApp] Converting notebook my_functions.ipynb to python
[NbConvertApp] Writing 24203 bytes to my_functions.py


<module 'my_functions' from 'C:\\Users\\ejohns\\Documents\\GitHub\\2025-Dinh-Lab-Research-Project\\Shapiro 2025 Project\\Python SquidPy\\my_functions.py'>

In [12]:
import my_functions
def mininum_distance_exclusive_pairing(distance_matrix):
    import pandas as pd
    import numpy as np
    from scipy.optimize import linear_sum_assignment
    
    # --- 1. Define your Distance Matrix as a Pandas DataFrame ---
    distance_data=distance_matrix
    distance_df = pd.DataFrame(distance_data)

    display(distance_matrix)
    
    # --- 2. Apply the Hungarian Algorithm ---
    # The scipy function requires a NumPy array, so we extract it using .to_numpy()
    cost_matrix = distance_df.to_numpy()
    pmn_indices, target_indices = linear_sum_assignment(cost_matrix)

    display(pmn_indices)
    display(target_indices)
    
    # --- 3. Map Indices back to DataFrame Labels ---
    # Get the actual labels from the DataFrame's index and columns
    paired_pmns = distance_df.index[pmn_indices]
    paired_targets = distance_df.columns[target_indices]
    
    # Get the distances for the optimal pairs
    optimal_distances = cost_matrix[pmn_indices, target_indices]
    
    # --- 4. Create a DataFrame for the Results and Display ---
    # This provides a clean, readable output of the optimal pairings.
    results_df = pd.DataFrame({
        'PMN_Cell': paired_pmns,
        'Paired_Target': paired_targets,
        'Distance': optimal_distances
    })
    
    average_distance = results_df['Distance'].mean()
    print(f'The average distance is: {average_distance}')
    
    # Identify unpaired PMNs using the DataFrame's index
    all_pmns = set(distance_df.index)
    paired_pmns_set = set(paired_pmns)
    unpaired_pmns = all_pmns - paired_pmns_set

    return{
        "Pair Matrix":results_df,
        "Average Distance":average_distance
    }


sample_id="c_1_6"
target_cell="CD8+T_cells"

# Gets Distance Matrix Between PMNs and Target Cells
results=my_functions.nearest_cells_of_particular_type(adata,sample_id,target_cell)
distance_matrix=results["Distance Matrix"]
distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)

results=mininum_distance_exclusive_pairing(distance_matrix)
display(results["Pair Matrix"])
print(f'The average distance is: {results["Average Distance"]}')

,c_1_6_1898,c_1_6_1902,c_1_6_1914
c_1_6_205,683.180547,640.775213,358.345733
c_1_6_269,681.989063,639.860306,360.845854
c_1_6_275,690.260543,648.102848,368.643798
c_1_6_294,689.775533,647.789897,370.429676
c_1_6_316,694.120316,652.262569,376.389968
c_1_6_423,703.603034,662.344235,393.427736
c_1_6_587,723.420894,683.492985,429.231353
c_1_6_623,728.987214,689.280108,437.271341
c_1_6_863,769.459808,732.013497,502.225287
c_1_6_2023,682.171546,639.353903,351.941516


array([10, 12, 14], dtype=int64)

array([2, 1, 0], dtype=int64)

The average distance is: 554.3990149132806


,PMN_Cell,Paired_Target,Distance
0,c_1_6_2074,c_1_6_1914,350.446711
1,c_1_6_2161,c_1_6_1902,634.393627
2,c_1_6_2250,c_1_6_1898,678.356707


The average distance is: 554.3990149132806


In [20]:
# Output. Each sample in order with the following columns Sample ID, PMN Count, Resonder Status, Average Min Distance CD4, Average Min Distance  CD8,  Average Min Distance  tumor cells,  Average Min Distance Treg (to start
import pandas as pd
import importlib
import my_functions
importlib.reload(my_functions)

# Define column names
columns = ['Sample ID', 'PMN Count', 'Responder Status', 'Avg Min CD4 Distance', 'Avg Min CD8 Distance', 'Avg Min Treg Distance', 'Average Min Tumor Distance', 'Avg CD4 Distance', 'Avg CD8 Distance', 'Avg Treg Distance', 'Average Tumor Distance', "CD4 Paired Average", "CD8 Paired Average", "Treg Paired Average", "Tumor Paired Average"]

# Create empty DataFrame with those columns
summary_matrix = pd.DataFrame(columns=columns)

for i in range(1,4):
    for j in range(1,26):
        
        # Data to add to matrix
        sample_id=f"c_{i}_{j}_"
        pmn_count=my_functions.pmn_counter(adata,sample_id)["PMN Count"]

        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue
        responder_status=my_functions.get_sample_info(adata,sample_id)["Response"]

        # CD4 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD4+T_cells")["Cell Count"] > 0:
            results_CD4=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD4+T_cells")
            avg_min_CD4=results_CD4["Average Mininum Distance"]
            avg_CD4=results_CD4["Average Distance"]

            #Paired Math
            distance_matrix=results_CD4["Distance Matrix"]
            distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
            paired_avg_min_CD4=mininum_distance_exclusive_pairing(distance_matrix)["Average Distance"]
        else:
            avg_min_CD4=0
            avg_CD4=0
            paired_avg_min_CD4=0

        # CD8 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD8+T_cells")["Cell Count"] > 0:
            results_CD8=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD8+T_cells")
            avg_min_CD8=results_CD8["Average Mininum Distance"]
            avg_CD8=results_CD8["Average Distance"]


            #Paired Math
            distance_matrix=results_CD8["Distance Matrix"]
            distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
            paired_avg_min_CD8=mininum_distance_exclusive_pairing(distance_matrix)["Average Distance"]
        else:
            avg_min_CD8=0
            avg_CD8=0
            paired_avg_min_CD8=0

        # Treg Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Treg")["Cell Count"] > 0:
            results_Treg=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Treg")
            avg_min_Treg=results_Treg["Average Mininum Distance"]
            avg_Treg=results_Treg["Average Distance"]

            #Paired Math
            distance_matrix=results_Treg["Distance Matrix"]
            distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
            paired_avg_min_Treg=mininum_distance_exclusive_pairing(distance_matrix)["Average Distance"]
        else:
            avg_min_Treg=0
            avg_Treg=0
            paired_avg_min_Treg=0

        # Tumor Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Tumor_cells")["Cell Count"] > 0:
            results_Tumor=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Tumor_cells")
            avg_min_tumor=results_Tumor["Average Mininum Distance"]
            avg_tumor=results_Tumor["Average Distance"]

            #Paired Math
            distance_matrix=results_Tumor["Distance Matrix"]
            distance_matrix=distance_matrix.drop(["Minimum Distance","Average Distance"],axis=1)
            paired_avg_min_Tumor=mininum_distance_exclusive_pairing(distance_matrix)["Average Distance"]
        else:
            avg_min_tumor=0
            avg_tumor=0
            paired_avg_min_Tumor=0
            
        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue

        # Adding data to matrix
        summary_matrix.loc[len(summary_matrix)] = [sample_id, pmn_count, responder_status, avg_min_CD4, avg_min_CD8, avg_min_Treg, avg_min_tumor, avg_CD4, avg_CD8, avg_Treg, avg_tumor,paired_avg_min_CD4, paired_avg_min_CD8, paired_avg_min_Treg, paired_avg_min_Tumor]
        
display(summary_matrix)
summary_matrix.to_excel("summary_matrix_7_1.xlsx", index=False)

[NbConvertApp] Converting notebook my_functions.ipynb to python
[NbConvertApp] Writing 24127 bytes to my_functions.py


The average distance is: 436.8167220452529

Minimum Overall Distance: 3931.350498407276
The average distance is: 647.5128962851905

Minimum Overall Distance: 9712.693444277858
The average distance is: 617.5828916077229

Minimum Overall Distance: 1235.1657832154458
The average distance is: 16.299001904816624

Minimum Overall Distance: 896.4451047649144
The average distance is: 44.325139628945344

Minimum Overall Distance: 265.95083777367205
The average distance is: 193.6821596510591

Minimum Overall Distance: 1162.0929579063545
The average distance is: 148.42625646371744

Minimum Overall Distance: 742.1312823185872
The average distance is: 11.824233477580636

Minimum Overall Distance: 70.94540086548382
The average distance is: 54.46832437090067

Minimum Overall Distance: 653.619892450808
The average distance is: 104.49721940106825

Minimum Overall Distance: 1253.966632812819
The average distance is: 111.90908610160211

Minimum Overall Distance: 1342.9090332192254
The average distance is

,Sample ID,PMN Count,Responder Status,Avg Min CD4 Distance,Avg Min CD8 Distance,Avg Min Treg Distance,Average Min Tumor Distance,Avg CD4 Distance,Avg CD8 Distance,Avg Treg Distance,Average Tumor Distance,CD4 Paired Average,CD8 Paired Average,Treg Paired Average,Tumor Paired Average
0,c_1_1_,55,R,273.972614,592.759764,631.634319,11.090332,525.352298,735.917035,651.116774,446.532664,436.816722,647.512896,617.582892,16.299002
1,c_1_3_,6,R,28.753049,170.031600,106.209140,11.824233,246.332199,351.629350,224.877578,330.041189,44.325140,193.682160,148.426256,11.824233
2,c_1_4_,12,R,52.528240,99.802765,73.496580,23.712366,234.948735,287.666555,248.699919,369.395953,54.468324,104.497219,111.909086,28.402609
3,c_1_5_,15,R,86.062535,150.281274,64.924672,15.375530,347.959190,392.597713,366.510778,400.521354,114.141089,272.331172,88.089597,17.137522
4,c_1_6_,23,R,19.640478,375.802924,74.626134,33.386715,99.011878,573.904537,74.626134,496.276498,11.740742,554.399015,6.106055,50.561473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,c_3_16_,18,NR,75.778988,183.079156,64.614820,29.833396,302.747329,264.613239,294.534193,463.074163,110.675053,55.367685,71.377941,35.208807
62,c_3_17_,22,NR,63.788160,151.555074,100.891480,30.778431,336.877326,260.473369,317.286085,496.839089,130.807960,52.510197,141.687419,44.300969
63,c_3_18_,36,NR,55.022038,144.141265,64.205251,28.126310,419.253659,459.742630,409.799740,474.083958,78.218014,308.913761,157.056179,30.249861
64,c_3_19_,36,NR,48.176947,348.449298,62.653424,22.721432,370.779279,513.769397,328.502648,404.308327,232.189566,289.403408,150.448976,23.839213
